In [ ]:
from pathlib import Path

import config
from utils.compare_fs_vs_baseline import build_comparison
from utils.export_metrics_to_csv import save_csv

%load_ext autoreload
%autoreload 2

In [ ]:
# === CELULA DE CONFIGURACAO -- Tarefa 6-gate (mesmo padrao de
# notebook/compare_fs_results.ipynb, Tarefa 3.2) ===
# Versao para a familia MLP single (SKlearnModel) -- baseline_dir continua
# SEMPRE data/result/chamados (os 5 baselines intocaveis da Secao 3 do
# CLAUDE.md), mas o MODELO baseline dentro dele e '1mlp', nao '1amv1'
# (familia hibrida ARIMA-MLP). build_comparison() foi generalizada nesta
# tarefa para aceitar baseline_model_name/linear_model_name_to_exclude como
# parametros -- reuso direto para SVR single/ARIMA-SVR depois, sem duplicar
# a logica de comparacao (so este bloco de configuracao muda por familia).
baseline_dir = Path(config.MODEL_DATA_PATH) / 'chamados'
baseline_model_name = '1mlp'
# MLP single (SKlearnModel) nao copia nenhum modelo linear pre-treinado para
# dentro do result_dir de FS (ao contrario do hibrido, que precisa do ARIMA
# sob o mesmo experiment_id) -- None desliga esse filtro por completo.
linear_model_name_to_exclude = None

# (rotulo, experiment_id) de cada variante de FS a comparar -- rotulo vira o
# prefixo das colunas no DataFrame de saida (<rotulo>_RMSE, <rotulo>_PctGain,
# <rotulo>_NFeatures).
fs_variants = [
    ('ftest', 'chamados_v4_fs_mlp_ftest'),
    ('mutualinfo', 'chamados_v4_fs_mlp_mutualinfo'),
    ('rfembedded', 'chamados_v4_fs_mlp_rfembedded'),
    ('lasso', 'chamados_v4_fs_mlp_lasso'),
    ('rfecv', 'chamados_v4_fs_mlp_rfecv'),
]

output_path = Path(config.ROOT_PATH) / 'results' / 'chamados_v4_fs_mlp_comparison.csv'

In [ ]:
# Compara o baseline x cada variante de FS lado a lado, por serie -- mesma
# funcao usada pela CLI (src/utils/compare_fs_vs_baseline.py) e pelo
# notebook irmao da familia hibrida (compare_fs_results.ipynb), so o
# baseline_model_name/linear_model_name_to_exclude mudam (celula 1).
fs_dirs = {rotulo: Path(config.MODEL_DATA_PATH) / experiment_id for rotulo, experiment_id in fs_variants}

df_comparison = build_comparison(
    baseline_dir,
    fs_dirs,
    baseline_model_name=baseline_model_name,
    linear_model_name_to_exclude=linear_model_name_to_exclude,
)
save_csv(df_comparison, output_path, label='CSV de comparacao MLP single (baseline x variantes FS)')
df_comparison